# Baby Step 6 — Design Remedies, Procedural Options, Damages Scenarios, and Settlement Architecture

**Author:** Alejandro Reynoso  
**Persistent vault:** `/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain`

Baby Step 6 converts the refreshed evidence base into a comparable strategy-design problem.

For each active matter, the notebook:

- models procedural options;
- models remedy alternatives;
- estimates low, expected, and high exposure;
- develops settlement ranges;
- compares strategy trade-offs;
- runs sensitivity analysis;
- selects an internal working case;
- records DEC-006;
- preserves Recommendation V1;
- does not create Recommendation V2.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, csv, datetime, statistics, math
from collections import defaultdict, Counter

VAULT = Path(r"/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain")

if not VAULT.exists():
    raise FileNotFoundError("Run Baby Step 0 first.")

state_path = VAULT/"00_System"/"Workflow_State.json"
state = json.loads(state_path.read_text(encoding="utf-8"))

if 5 not in state.get("completed_steps", []):
    raise RuntimeError("Baby Step 5 is not complete.")

matters = json.loads((VAULT/"data"/"active_matters.json").read_text(encoding="utf-8"))
recommendations = json.loads((VAULT/"data"/"baby_step_1_recommendations_v1.json").read_text(encoding="utf-8"))
refreshed_permissions = json.loads((VAULT/"data"/"baby_step_5_refreshed_permission_state.json").read_text(encoding="utf-8"))
refreshed_contradictions = json.loads((VAULT/"data"/"baby_step_5_refreshed_contradictions.json").read_text(encoding="utf-8"))
new_claims = json.loads((VAULT/"data"/"baby_step_3_claims.json").read_text(encoding="utf-8"))

print("Matters:", len(matters))
print("Recommendation V1 records:", len(recommendations))


## Strategy-design principle

Baby Step 6 does not ask for a universal “best strategy.”

It compares alternatives against:

- legal viability;
- evidentiary readiness;
- expected remedy;
- timing;
- cost;
- downside exposure;
- business disruption;
- settlement leverage;
- reversibility;
- governance risk.

The selected output is an **internal working case**, not an external legal instruction.


In [ ]:
STRATEGY_OPTIONS = {
    "MAT-001": [
        "Preliminary injunction",
        "Expedited declaratory relief",
        "Negotiated consent or buyout",
        "Full merits litigation"
    ],
    "MAT-002": [
        "Expert determination",
        "Early contract interpretation ruling",
        "Accounting mediation",
        "Full litigation"
    ],
    "MAT-003": [
        "Targeted injunction and technical discovery",
        "Royalty audit and damages claim",
        "License renegotiation",
        "Full merits litigation"
    ],
    "MAT-004": [
        "Contractual buy-sell process",
        "Judicial dissolution or equitable relief",
        "Governance reset and staged buyout",
        "Full litigation"
    ],
    "MAT-005": [
        "Partial summary judgment",
        "Liability-limitation and causation defense",
        "Commercial continuity settlement",
        "Full trial preparation"
    ]
}

assert set(STRATEGY_OPTIONS) == {m["matter_id"] for m in matters}


## Evaluation dimensions

Each option receives a transparent score across ten dimensions.


In [ ]:
DIMENSION_WEIGHTS = {
    "legal_viability": 0.16,
    "evidence_readiness": 0.12,
    "remedy_fit": 0.12,
    "timing_efficiency": 0.10,
    "cost_efficiency": 0.08,
    "downside_control": 0.12,
    "business_continuity": 0.10,
    "settlement_leverage": 0.08,
    "reversibility": 0.06,
    "governance_alignment": 0.06
}

assert abs(sum(DIMENSION_WEIGHTS.values()) - 1.0) < 1e-9

(VAULT/"00_System"/"Baby_Step_6_Strategy_Design_Model.json").write_text(
    json.dumps({
        "weights": DIMENSION_WEIGHTS,
        "purpose": "internal remedies and strategy comparison",
        "not_for": [
            "external legal advice",
            "court prediction",
            "settlement authority",
            "filing instruction"
        ]
    }, indent=2),
    encoding="utf-8"
)


In [ ]:
def base_scores(matter_id, option_index):
    matrices = {
        "MAT-001": [
            [86,78,92,84,60,72,48,88,42,80],
            [82,74,78,72,66,70,72,74,68,76],
            [70,68,74,78,82,86,92,80,88,90],
            [80,82,88,35,28,58,30,70,20,72]
        ],
        "MAT-002": [
            [90,88,84,82,80,82,86,78,72,88],
            [84,76,80,60,58,70,72,82,58,80],
            [76,80,72,78,76,80,90,84,86,82],
            [78,84,84,34,24,55,44,72,18,74]
        ],
        "MAT-003": [
            [84,86,92,82,52,68,46,88,40,80],
            [82,88,78,68,70,72,76,80,66,78],
            [68,74,72,80,82,88,94,78,90,90],
            [80,90,86,32,22,52,38,68,16,72]
        ],
        "MAT-004": [
            [88,82,86,80,74,82,76,84,70,92],
            [78,76,90,42,30,58,36,72,22,76],
            [82,80,84,76,68,88,94,86,82,96],
            [76,84,88,28,20,50,30,66,14,70]
        ],
        "MAT-005": [
            [82,84,78,72,68,82,70,86,64,80],
            [88,86,82,76,80,90,78,88,72,84],
            [72,78,74,84,86,86,96,80,90,88],
            [80,88,86,30,20,54,32,64,12,72]
        ]
    }
    values = matrices[matter_id][option_index]
    keys = list(DIMENSION_WEIGHTS.keys())
    return dict(zip(keys, values))

evaluations = {}

for matter in matters:
    mid = matter["matter_id"]
    options = []

    for idx, option in enumerate(STRATEGY_OPTIONS[mid]):
        scores = base_scores(mid, idx)
        weighted = {
            k: round(scores[k] * DIMENSION_WEIGHTS[k], 4)
            for k in DIMENSION_WEIGHTS
        }
        total = round(sum(weighted.values()), 2)

        options.append({
            "matter_id": mid,
            "strategy_option": option,
            "dimension_scores": scores,
            "weighted_contributions": weighted,
            "strategy_design_score": total
        })

    options.sort(key=lambda x: -x["strategy_design_score"])
    evaluations[mid] = options

(VAULT/"data"/"baby_step_6_strategy_option_evaluations.json").write_text(
    json.dumps(evaluations, indent=2),
    encoding="utf-8"
)

for mid, opts in evaluations.items():
    print(mid, opts[0]["strategy_option"], opts[0]["strategy_design_score"])


## Damages and exposure scenarios

Each matter receives low, expected, and high exposure scenarios derived from:

- claimed damages;
- modeled exposure;
- evidence confidence;
- contradiction status;
- likely remedy limitations;
- procedural risk.

These are synthetic analytical scenarios, not accounting reserves.


In [ ]:
scenario_records = []

for matter in matters:
    mid = matter["matter_id"]
    permission = next(
        p for p in refreshed_permissions
        if p["matter_id"] == mid
    )

    confidence = permission["refreshed_average_claim_confidence"]
    claimed = matter["claimed_damages_usd"]
    modeled = matter["modeled_exposure_usd"]

    unresolved = permission["remaining_high_severity_contradictions"]
    narrowed = permission["narrowed_contradictions"]

    uncertainty_multiplier = 1 + 0.08*unresolved + 0.03*narrowed
    confidence_factor = max(0.55, min(0.95, confidence/100))

    expected = round(modeled * confidence_factor)
    low = round(expected * 0.55)
    high = round(min(claimed, expected * 1.45 * uncertainty_multiplier))

    scenario_records.append({
        "matter_id": mid,
        "claimed_damages_usd": claimed,
        "baseline_modeled_exposure_usd": modeled,
        "low_scenario_usd": low,
        "expected_scenario_usd": expected,
        "high_scenario_usd": high,
        "refreshed_claim_confidence": confidence,
        "unresolved_high_contradictions": unresolved,
        "narrowed_contradictions": narrowed,
        "synthetic": True
    })

(VAULT/"data"/"baby_step_6_damages_scenarios.json").write_text(
    json.dumps(scenario_records, indent=2),
    encoding="utf-8"
)

print(json.dumps(scenario_records, indent=2))


## Settlement architecture

Each matter receives a synthetic settlement floor, midpoint, ceiling, and non-monetary term set.

The settlement range is not authority to negotiate. It is an internal decision-analysis tool.


In [ ]:
NON_MONETARY_TERMS = {
    "MAT-001": [
        "governance protections",
        "consent protocol",
        "transfer restrictions",
        "board representation"
    ],
    "MAT-002": [
        "accounting methodology clarification",
        "expert-selection protocol",
        "release language",
        "future earnout governance"
    ],
    "MAT-003": [
        "license-scope reset",
        "audit rights",
        "confidentiality controls",
        "future royalty framework"
    ],
    "MAT-004": [
        "governance reset",
        "staged buyout",
        "deadlock escalation",
        "operational covenants"
    ],
    "MAT-005": [
        "supply continuity",
        "pricing adjustment",
        "service-level commitments",
        "damages release"
    ]
}

settlement_records = []

for scenario in scenario_records:
    mid = scenario["matter_id"]
    expected = scenario["expected_scenario_usd"]
    low = scenario["low_scenario_usd"]
    high = scenario["high_scenario_usd"]

    floor = round(max(low*0.80, expected*0.45))
    midpoint = round(expected*0.78)
    ceiling = round(min(high, expected*1.10))

    settlement_records.append({
        "matter_id": mid,
        "settlement_floor_usd": floor,
        "settlement_midpoint_usd": midpoint,
        "settlement_ceiling_usd": ceiling,
        "non_monetary_terms": NON_MONETARY_TERMS[mid],
        "authority_status": "INTERNAL ANALYTICAL RANGE ONLY",
        "synthetic": True
    })

(VAULT/"data"/"baby_step_6_settlement_architecture.json").write_text(
    json.dumps(settlement_records, indent=2),
    encoding="utf-8"
)


## Sensitivity analysis

The strategy-design model is stressed by changing:

- legal-viability weight;
- evidence-readiness weight;
- downside-control weight;
- business-continuity weight;
- settlement-leverage weight.

The notebook identifies whether the preferred working case is robust or fragile.


In [ ]:
SENSITIVITY_DIMENSIONS = [
    "legal_viability",
    "evidence_readiness",
    "downside_control",
    "business_continuity",
    "settlement_leverage"
]

def perturb(weights, target, factor):
    updated = dict(weights)
    updated[target] *= factor
    total = sum(updated.values())
    return {k:v/total for k,v in updated.items()}

def score_option(option, weights):
    return round(sum(
        option["dimension_scores"][k] * weights[k]
        for k in weights
    ), 2)

sensitivity_results = {}

for matter in matters:
    mid = matter["matter_id"]
    base_top = evaluations[mid][0]["strategy_option"]
    tests = []

    for dimension in SENSITIVITY_DIMENSIONS:
        for factor in [0.80, 1.20]:
            weights = perturb(DIMENSION_WEIGHTS, dimension, factor)
            ranked = sorted(
                [
                    {
                        "strategy_option": option["strategy_option"],
                        "score": score_option(option, weights)
                    }
                    for option in evaluations[mid]
                ],
                key=lambda x: -x["score"]
            )
            tests.append({
                "dimension": dimension,
                "factor": factor,
                "preferred_strategy": ranked[0]["strategy_option"],
                "preferred_changed": ranked[0]["strategy_option"] != base_top,
                "top_score": ranked[0]["score"]
            })

    switch_count = sum(1 for t in tests if t["preferred_changed"])
    fragility = round(100*switch_count/len(tests), 2)

    sensitivity_results[mid] = {
        "matter_id": mid,
        "base_preferred_strategy": base_top,
        "tests": tests,
        "switch_count": switch_count,
        "fragility_score": fragility
    }

(VAULT/"data"/"baby_step_6_strategy_sensitivity.json").write_text(
    json.dumps(sensitivity_results, indent=2),
    encoding="utf-8"
)

for mid, result in sensitivity_results.items():
    print(mid, result["fragility_score"])


## Internal working case

Each matter receives one internal working case, one fallback, and one preserved rejected alternative.

This is not Recommendation V2.


In [ ]:
working_cases = []

for matter in matters:
    mid = matter["matter_id"]
    ranked = evaluations[mid]
    scenario = next(x for x in scenario_records if x["matter_id"] == mid)
    settlement = next(x for x in settlement_records if x["matter_id"] == mid)
    sensitivity = sensitivity_results[mid]

    working_cases.append({
        "working_case_id": f"WC-{mid}-BS6",
        "matter_id": mid,
        "selected_strategy": ranked[0]["strategy_option"],
        "fallback_strategy": ranked[1]["strategy_option"],
        "preserved_rejected_alternatives": [
            ranked[2]["strategy_option"],
            ranked[3]["strategy_option"]
        ],
        "strategy_design_score": ranked[0]["strategy_design_score"],
        "fragility_score": sensitivity["fragility_score"],
        "low_exposure_usd": scenario["low_scenario_usd"],
        "expected_exposure_usd": scenario["expected_scenario_usd"],
        "high_exposure_usd": scenario["high_scenario_usd"],
        "settlement_floor_usd": settlement["settlement_floor_usd"],
        "settlement_midpoint_usd": settlement["settlement_midpoint_usd"],
        "settlement_ceiling_usd": settlement["settlement_ceiling_usd"],
        "non_monetary_terms": settlement["non_monetary_terms"],
        "recommendation_v2_created": False,
        "authority_status": "INTERNAL WORKING CASE",
        "synthetic": True
    })

(VAULT/"data"/"baby_step_6_working_cases.json").write_text(
    json.dumps(working_cases, indent=2),
    encoding="utf-8"
)

print(json.dumps(working_cases, indent=2))


In [ ]:
def write_note(path, lines):
    path.write_text("\n".join(lines).strip()+"\n", encoding="utf-8")

strategy_dir = VAULT/"18_Strategy_Design"
strategy_dir.mkdir(parents=True, exist_ok=True)

for working in working_cases:
    mid = working["matter_id"]
    matter = next(m for m in matters if m["matter_id"] == mid)
    options = evaluations[mid]

    lines = [
        "---",
        f"working_case_id: {working['working_case_id']}",
        f"matter_id: {mid}",
        "baby_step: 6",
        "recommendation_v2_created: false",
        "authority_status: INTERNAL WORKING CASE",
        "synthetic: true",
        "---","",
        f"# {working['working_case_id']} — Strategy Design","",
        f"- Matter: [[../02_Active_Matters/{mid}]]",
        f"- Recommendation V1: [[../08_Recommendations/REC-{mid}-V001]]","",
        "## Selected working case","",
        f"**{working['selected_strategy']}**","",
        "## Fallback strategy","",
        working["fallback_strategy"],"",
        "## Preserved rejected alternatives",""
    ]
    lines += [f"- {x}" for x in working["preserved_rejected_alternatives"]]
    lines += [
        "","## Exposure scenarios","",
        f"- Low: ${working['low_exposure_usd']:,}",
        f"- Expected: ${working['expected_exposure_usd']:,}",
        f"- High: ${working['high_exposure_usd']:,}","",
        "## Settlement architecture","",
        f"- Floor: ${working['settlement_floor_usd']:,}",
        f"- Midpoint: ${working['settlement_midpoint_usd']:,}",
        f"- Ceiling: ${working['settlement_ceiling_usd']:,}","",
        "### Non-monetary terms",""
    ]
    lines += [f"- {x}" for x in working["non_monetary_terms"]]
    lines += [
        "","## Strategy ranking","",
    ]
    for idx, option in enumerate(options, start=1):
        lines.append(
            f"{idx}. {option['strategy_option']} — "
            f"{option['strategy_design_score']}/100"
        )
    lines += [
        "","## Sensitivity","",
        f"Fragility score: {working['fragility_score']}/100","",
        "## Governance","",
        "This working case is not Recommendation V2 and creates no external authority."
    ]
    write_note(strategy_dir/f"{working['working_case_id']}.md", lines)

print("Strategy-design notes:", len(list(strategy_dir.glob("*.md"))))


## Strategy-design memorandum

The portfolio memorandum compares all five working cases and makes the trade-offs visible.


In [ ]:
memo = [
    "# Baby Step 6 — Remedies, Damages, and Strategy-Design Memorandum","",
    "## Executive conclusion","",
    "The refreshed evidence base now supports a structured comparison of remedies, procedural alternatives, exposure scenarios, and settlement architectures.",
    "The outputs are internal working cases only and do not replace Recommendation V1.","",
    "## Portfolio working cases","",
    "| Matter | Selected strategy | Score | Fragility | Expected exposure | Settlement midpoint |",
    "|---|---|---:|---:|---:|---:|"
]

for working in working_cases:
    memo.append(
        f"| [[../02_Active_Matters/{working['matter_id']}]] | "
        f"{working['selected_strategy']} | "
        f"{working['strategy_design_score']} | "
        f"{working['fragility_score']} | "
        f"${working['expected_exposure_usd']:,} | "
        f"${working['settlement_midpoint_usd']:,} |"
    )

memo += [
    "","## Governance conclusion","",
    "The strategy-design layer may support internal planning and future recommendation review.",
    "It does not create Recommendation V2, settlement authority, filing authority, or external communication authority."
]

write_note(
    VAULT/"10_Reports"/"Baby_Step_6_Strategy_Design_Memorandum.md",
    memo
)


## Human decision — DEC-006

DEC-006 accepts the five internal working cases as the remedies-and-strategy baseline.

It authorizes:

- synthetic counterparty and expert selection;
- conflict screening;
- internal process design;
- continued sensitivity analysis.

It does not authorize Recommendation V2 or external action.


In [ ]:
DECISION = {
    "decision_id": "DEC-006",
    "date": datetime.date.today().isoformat(),
    "title": "Accept Remedies and Strategy-Design Working Cases",
    "decision": (
        "Accept the five Baby Step 6 working cases as the internal "
        "remedies, exposure, and settlement-design baseline."
    ),
    "accepted_working_cases": [
        w["working_case_id"] for w in working_cases
    ],
    "permitted_next_actions": [
        "synthetic outside-counsel and expert selection",
        "conflict screening",
        "internal process design",
        "continued sensitivity analysis",
        "preserve Recommendation V1"
    ],
    "not_authorized": [
        "Recommendation V2",
        "filing",
        "service",
        "party contact",
        "court contact",
        "external counsel instruction",
        "settlement offer",
        "external legal advice",
        "external distribution"
    ],
    "synthetic": True
}

(VAULT/"09_Decisions"/"DEC-006.json").write_text(
    json.dumps(DECISION, indent=2),
    encoding="utf-8"
)

lines = [
    "# DEC-006 — Accept Remedies and Strategy-Design Working Cases","",
    f"**Date:** {DECISION['date']}","","## Decision","",DECISION["decision"],"",
    "## Accepted working cases",""
]
lines += [f"- [[../18_Strategy_Design/{wc}]]" for wc in DECISION["accepted_working_cases"]]
lines += ["","## Permitted next actions",""]
lines += [f"- {x}" for x in DECISION["permitted_next_actions"]]
lines += ["","## Not authorized",""]
lines += [f"- {x}" for x in DECISION["not_authorized"]]

write_note(VAULT/"09_Decisions"/"DEC-006.md", lines)


In [ ]:
hot = [
    "# Current State — Hot Cache","",
    "## Recommendation state","",
    "- Recommendation V1 remains preserved.",
    "- Recommendation V2 does not exist.","",
    "## Strategy-design state",""
]
hot += [
    f"- {w['matter_id']}: [[../18_Strategy_Design/{w['working_case_id']}]] — "
    f"{w['selected_strategy']}"
    for w in working_cases
]
hot += [
    "","## Current decision","","- [[../09_Decisions/DEC-006]]","",
    "## Permitted","",
    "- Synthetic outside-counsel and expert selection",
    "- Conflict screening",
    "- Internal process design",
    "- Continued sensitivity analysis","",
    "## Prohibited","",
    "- Recommendation V2",
    "- Filing or service",
    "- Party or court contact",
    "- External counsel instruction",
    "- Settlement offers",
    "- External legal advice","",
    "## Next permitted experiment","",
    "Select synthetic outside counsel, experts, and vendors using conflict-aware gates."
]

write_note(VAULT/"12_Hot_Cache"/"Current_State.md", hot)


In [ ]:
errors = []

strategy_notes = list((VAULT/"18_Strategy_Design").glob("*.md"))
v1 = list((VAULT/"08_Recommendations").glob("REC-*-V001.md"))
v2 = list((VAULT/"08_Recommendations").glob("REC-*-V002.md"))

if len(strategy_notes) != 5:
    errors.append(f"Expected 5 strategy-design notes, found {len(strategy_notes)}")

if len(v1) != 5:
    errors.append(f"Expected 5 Recommendation V1 notes, found {len(v1)}")

if v2:
    errors.append("Recommendation V2 exists prematurely")

if len(working_cases) != 5:
    errors.append("Missing working cases")

required = [
    VAULT/"data"/"baby_step_6_strategy_option_evaluations.json",
    VAULT/"data"/"baby_step_6_damages_scenarios.json",
    VAULT/"data"/"baby_step_6_settlement_architecture.json",
    VAULT/"data"/"baby_step_6_strategy_sensitivity.json",
    VAULT/"data"/"baby_step_6_working_cases.json",
    VAULT/"10_Reports"/"Baby_Step_6_Strategy_Design_Memorandum.md",
    VAULT/"09_Decisions"/"DEC-006.md",
    VAULT/"09_Decisions"/"DEC-006.json"
]

for path in required:
    if not path.exists():
        errors.append(f"Missing required output: {path}")

validation = {
    "validated_at": datetime.datetime.now().isoformat(),
    "working_case_count": len(working_cases),
    "strategy_note_count": len(strategy_notes),
    "recommendation_v1_count": len(v1),
    "recommendation_v2_count": len(v2),
    "decision": "DEC-006",
    "errors": errors,
    "passed": len(errors) == 0
}

(VAULT/"11_Audit"/"Baby_Step_6_Validation.json").write_text(
    json.dumps(validation, indent=2),
    encoding="utf-8"
)

assert validation["passed"], errors
print(json.dumps(validation, indent=2))
print("BABY STEP 6 PASSED")


In [ ]:
state.update({
    "completed_steps": sorted(set(state.get("completed_steps", []) + [6])),
    "current_step": 6,
    "next_step": 7,
    "decision": "DEC-006",
    "strategy_working_cases": 5,
    "current_recommendation_version": 1,
    "next_problem": (
        "Select synthetic outside counsel, experts, and vendors "
        "using conflict-aware gates."
    ),
    "permission_state": {
        "observe": True,
        "organize": True,
        "browse": True,
        "internal_strategy_analysis": True,
        "evidence_governance": True,
        "committee_product": True,
        "controlled_internal_diligence": True,
        "remedies_and_damages_analysis": True,
        "counterparty_and_expert_selection": True,
        "recommendation_v2": False,
        "external_action": False
    }
})

state_path.write_text(
    json.dumps(state, indent=2),
    encoding="utf-8"
)

audit = {
    "timestamp": datetime.datetime.now().isoformat(),
    "step": 6,
    "action": (
        "Designed remedies, procedural options, exposure scenarios, "
        "settlement architecture, and internal working cases."
    ),
    "outputs": {
        "working_cases": len(working_cases),
        "decision": "DEC-006"
    },
    "validation_passed": True
}

with (VAULT/"11_Audit"/"workflow_audit.jsonl").open("a", encoding="utf-8") as f:
    f.write(json.dumps(audit) + "\n")
